# Week 20 - MLOps CI/CD and Monitoring (Data Engineer Variant)

Last week you used MLflow and the Model Registry to ship a fraud-detection
model behind the endpoint `week19-fraud-endpoint`. This week we shift our
attention to what data engineers actually own in production: the PIPELINE
that feeds that model.

If the input distribution silently changes, the model still answers, just
wrongly. The data engineer is the first line of defense.

## What you will build today

1. A Delta Change Data Feed based freshness monitor that watches new rows
   arriving in `bread_academy.course_data.fraud_transactions` and computes
   how the new batch differs from a 30-day rolling baseline.
2. A PySpark data quality gate with four production-grade assertions, run
   before any downstream consumer touches the data.
3. A Delta audit log table at
   `bread_academy.student_work.pipeline_audit_log` so every pipeline run
   leaves a queryable trace you can join with MLflow runs later.
4. An automated retraining trigger that calls the Databricks Workflows
   REST API to start the ML engineer's retraining job when your data
   quality signals fire.

## Learning objectives

- Enable and read Delta Change Data Feed for pipeline observability.
- Express data quality checks as PySpark assertions you can run in CI.
- Write structured audit events to a Delta table.
- Trigger Databricks Workflows programmatically via REST.

## Environment Setup

**Platform**: Databricks (Runtime 15.4 LTS ML).

**Cluster**: A shared course cluster is provisioned for you. Confirm the
runtime in the cluster UI before attaching this notebook.

**Required libraries**: All pinned via `%pip install` in the next cell.

**Secrets**: We read `aws-access-key`, `aws-secret-key`, and
`databricks-token` from the `aws-course-creds` scope. The course operators
have already populated this scope.

**Unity Catalog permissions** (already granted to your group):
- USE CATALOG on `bread_academy`
- USE SCHEMA, SELECT on `bread_academy.course_data`
- USE SCHEMA, CREATE TABLE, MODIFY on `bread_academy.student_work`

In [ ]:
%pip install --quiet \
    "mlflow==2.16.2" \
    "boto3>=1.35,<2" \
    "requests>=2.31,<3"
dbutils.library.restartPython()

In [ ]:
from importlib.metadata import version
import os, json, time, uuid
from datetime import datetime, timedelta, timezone
import requests
import boto3
import mlflow
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, BooleanType, TimestampType

print("mlflow:", version("mlflow"))
print("boto3:", version("boto3"))
print("requests:", version("requests"))

In [ ]:
# AWS creds for any boto3 calls we may need (read the endpoint name, etc.)
os.environ["AWS_ACCESS_KEY_ID"] = dbutils.secrets.get(scope="aws-course-creds", key="aws-access-key")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope="aws-course-creds", key="aws-secret-key")
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

# Databricks token for the Workflows REST API call later
DATABRICKS_TOKEN = dbutils.secrets.get(scope="aws-course-creds", key="databricks-token")

# Workspace URL (returned without scheme)
DATABRICKS_HOST = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
print("Workspace:", DATABRICKS_HOST)

# Names we will reuse throughout
SOURCE_TABLE = "bread_academy.course_data.fraud_transactions"
AUDIT_TABLE = "bread_academy.student_work.pipeline_audit_log"
MLFLOW_EXP = "/Shared/bread_academy/week20_pipeline_health"
ENDPOINT_NAME = "week19-fraud-endpoint"  # consumer downstream, not modified here

In [ ]:
# Probe 1: source table is reachable
src_count = spark.sql(f"SELECT COUNT(*) AS c FROM {SOURCE_TABLE}").collect()[0]["c"]
print(f"Source table OK: {src_count:,} rows.")

# Probe 2: student_work schema is writable
try:
    spark.sql(
        "CREATE TABLE IF NOT EXISTS bread_academy.student_work._probe_w20 "
        "(id INT) USING DELTA"
    )
    spark.sql("DROP TABLE bread_academy.student_work._probe_w20")
    print("student_work schema: WRITE OK.")
except Exception as e:
    print("Ask your instructor to grant CREATE TABLE on bread_academy.student_work.")
    raise

# Probe 3: MLflow experiment is reachable
mlflow.set_experiment(MLFLOW_EXP)
print(f"MLflow experiment OK: {MLFLOW_EXP}")

## What Are We Building Today?

Imagine you are on call for the Bread Financial fraud platform. At 3am the
on-call ML engineer pages you: "the model is flagging twice as many txns as
yesterday, did something change upstream?" Without pipeline observability
your only answer is to start querying tables. With it, you already know:
the source table received 1.4M new rows in the last 6 hours (normal: 800k),
the merchant_category mix shifted hard toward category 42 (a new partner
went live), and the PSI score on `amount` is 0.31 (way above the 0.2 alert
threshold).

You did not write the model. You did not retrain it. But you SAVED the
model owner from a bad night because the pipeline told its own story.

That is what the next four topics build, end to end.

## Topic 1 - Delta Change Data Feed as a freshness monitor

### Why CDF over a periodic `SELECT COUNT(*)`?

A periodic count tells you how many rows exist at a point in time. CDF tells
you exactly which rows ARRIVED, in what order, with what values. You can
compare yesterday's slice against today's slice without touching historical
data, and you can do it incrementally (cheap) rather than full-scanning the
table (expensive).

### Enabling CDF

CDF must be enabled before any change is captured. Past versions of the
table are NOT retroactively recorded:

```python
spark.sql(
    "ALTER TABLE bread_academy.course_data.fraud_transactions "
    "SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)
```

### Reading the feed

```python
df_changes = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 12)
    .table("bread_academy.course_data.fraud_transactions")
)
```

The result has the original columns plus `_change_type`, `_commit_version`,
and `_commit_timestamp`. We will use those three columns to slice the
"last hour" of arrivals.

In [ ]:
# (Instructor pre-enabled CDF on the source table; this is idempotent.)
spark.sql(
    f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)"
)

# Find the latest committed version of the table.
latest_version = (
    spark.sql(f"DESCRIBE HISTORY {SOURCE_TABLE}")
    .agg(F.max("version").alias("v"))
    .collect()[0]["v"]
)
print("Latest version:", latest_version)

# Read changes from a few versions back so the demo has something to show.
start_v = max(latest_version - 5, 0)
df_changes = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", start_v)
    .table(SOURCE_TABLE)
)

display(
    df_changes
    .filter(F.col("_change_type") == "insert")
    .select("_commit_version", "_commit_timestamp", "amount", "is_fraud")
    .orderBy(F.desc("_commit_timestamp"))
    .limit(20)
)

## Lab 1 - Build a freshness monitor (10 minutes)

Build a function `freshness_snapshot(start_version)` that returns a single
Python dict with these keys:

- `rows_arrived` - count of inserts since `start_version`.
- `latest_commit_ts` - max `_commit_timestamp` seen (Python datetime).
- `fraud_rate_recent` - fraction of recent arrivals where `is_fraud == 1`.
- `fraud_rate_baseline` - overall fraud rate of the table.
- `fraud_rate_delta` - `fraud_rate_recent - fraud_rate_baseline`.

Then log all five values to MLflow under experiment `MLFLOW_EXP` as a
single run named `freshness-<timestamp>`.

You can reuse `start_v` from the demo cell as your starting version.

In [ ]:
def freshness_snapshot(start_version: int) -> dict:
    # YOUR CODE
    snapshot = None
    return snapshot

# Run it and log
mlflow.set_experiment(MLFLOW_EXP)
with mlflow.start_run(run_name=f"freshness-{int(time.time())}"):
    snap = None  # YOUR CODE
    # YOUR CODE: log each numeric value as a metric, log latest_commit_ts as a tag

print(snap)

In [ ]:
# SAFETY-NET for Lab 1. Run this only if you did NOT finish Lab 1.
if 'snap' not in dir() or snap is None:
    print("Using Lab 1 safety-net.")
    inserts = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_v)
        .table(SOURCE_TABLE)
        .filter(F.col("_change_type") == "insert")
    )
    agg = inserts.agg(
        F.count("*").alias("rows_arrived"),
        F.max("_commit_timestamp").alias("latest_commit_ts"),
        F.avg(F.col("is_fraud").cast("double")).alias("fraud_rate_recent"),
    ).collect()[0]
    baseline = spark.table(SOURCE_TABLE).agg(
        F.avg(F.col("is_fraud").cast("double")).alias("b")
    ).collect()[0]["b"]
    snap = {
        "rows_arrived": int(agg["rows_arrived"] or 0),
        "latest_commit_ts": agg["latest_commit_ts"],
        "fraud_rate_recent": float(agg["fraud_rate_recent"] or 0.0),
        "fraud_rate_baseline": float(baseline or 0.0),
        "fraud_rate_delta": float((agg["fraud_rate_recent"] or 0.0) - (baseline or 0.0)),
    }
    print(snap)

## Think About It

You enabled CDF on the source table. A teammate asks: "Can I get yesterday's
changes too? CDF should give me history, right?" What do you tell them?

(Hint: CDF captures changes from the moment it is enabled. Anything before
that is invisible to the feed. For history you would have to reconstruct
from `DESCRIBE HISTORY` and time travel queries, which is not the same
thing.)

## Topic 2 - A data quality gate before the model sees the data

The freshness monitor tells you WHAT changed. A data quality gate decides
whether the change is acceptable. If it is not, you HALT the pipeline so
the downstream consumer (the fraud model) never sees the bad batch.

We will use four checks that cover 80 percent of real production failures:

1. No nulls in `is_fraud` (the label column must always be present).
2. `amount > 0` on every row (no refunds, no test rows).
3. `transaction_date` within the last 90 days (no historical replays).
4. Fraud rate between 1 percent and 20 percent (class balance sanity).

We implement them as plain PySpark assertions because (a) they are easy to
read, (b) they run on the same engine as the data, and (c) they do not add
a fragile dependency on Great Expectations for a beginner audience.

In [ ]:
def assert_no_null_labels(df, col_name="is_fraud"):
    n_null = df.filter(F.col(col_name).isNull()).count()
    if n_null > 0:
        raise AssertionError(f"DQ FAIL: {n_null} null values in '{col_name}'")
    return {"check": "no_null_labels", "passed": True, "n_null": 0}

# Demo on the current source table
result = assert_no_null_labels(spark.table(SOURCE_TABLE))
print(result)

## Lab 2 - Implement the remaining 3 checks (10 minutes)

Following the pattern in the demo, implement three functions. Each must
return a dict with at least `check`, `passed`, and one diagnostic field
(e.g., `n_bad`, `rate`, `min_date`). Each must raise `AssertionError` with
a clear message on failure:

1. `assert_positive_amount(df)` - all rows must have `amount > 0`.
2. `assert_recent_dates(df, days=90)` - all `transaction_date` values within
   the last `days` days.
3. `assert_fraud_rate_in_band(df, lo=0.01, hi=0.20)` - overall fraud rate in
   `[lo, hi]`.

Then wrap all four in a `run_quality_gate(df)` function that runs them
sequentially, collects results in a list, and returns `(all_passed, results)`.
The gate must NOT short-circuit on the first failure - we want to see every
failing check in one shot.

In [ ]:
def assert_positive_amount(df):
    # YOUR CODE
    return None

def assert_recent_dates(df, days=90):
    # YOUR CODE
    return None

def assert_fraud_rate_in_band(df, lo=0.01, hi=0.20):
    # YOUR CODE
    return None

def run_quality_gate(df):
    # YOUR CODE
    return None, []

all_passed, gate_results = run_quality_gate(spark.table(SOURCE_TABLE))
print("ALL PASSED:", all_passed)
for r in gate_results:
    print(r)

In [ ]:
# SAFETY-NET for Lab 2.
if 'all_passed' not in dir() or all_passed is None:
    print("Using Lab 2 safety-net.")

    def _check(name, df_filter, diagnostic_value, passed):
        return {"check": name, "passed": bool(passed), "value": diagnostic_value}

    def assert_positive_amount(df):
        n_bad = df.filter(F.col("amount") <= 0).count()
        passed = (n_bad == 0)
        if not passed:
            return _check("positive_amount", n_bad, passed)
        return _check("positive_amount", 0, True)

    def assert_recent_dates(df, days=90):
        cutoff = datetime.now(timezone.utc) - timedelta(days=days)
        n_bad = df.filter(F.col("transaction_date") < F.lit(cutoff)).count()
        passed = (n_bad == 0)
        return _check("recent_dates", n_bad, passed)

    def assert_fraud_rate_in_band(df, lo=0.01, hi=0.20):
        rate = df.agg(F.avg(F.col("is_fraud").cast("double")).alias("r")).collect()[0]["r"] or 0.0
        passed = (lo <= rate <= hi)
        return _check("fraud_rate_band", round(rate, 4), passed)

    def run_quality_gate(df):
        results = [
            assert_no_null_labels(df),
            assert_positive_amount(df),
            assert_recent_dates(df),
            assert_fraud_rate_in_band(df),
        ]
        return all(r["passed"] for r in results), results

    all_passed, gate_results = run_quality_gate(spark.table(SOURCE_TABLE))
    print("ALL PASSED:", all_passed)
    for r in gate_results:
        print(r)

## Topic 3 - Structured audit log

Now we have two signals (freshness snapshot, quality gate) and no place to
put them. MLflow is fine for ad-hoc metric logging but it is awkward to JOIN
against other tables for analytics. So we add a second store: a Delta table
in `bread_academy.student_work.pipeline_audit_log` that mirrors every run.

### Schema we will write

| Column | Type | Meaning |
|--------|------|---------|
| run_id | STRING | UUID, one per pipeline invocation |
| run_timestamp | TIMESTAMP | When the run completed |
| rows_processed | LONG | Total rows seen this run |
| fraud_rate | DOUBLE | Observed fraud rate this run |
| psi_score | DOUBLE | PSI of `amount` against the 30-day baseline |
| drift_detected | BOOLEAN | True if any DQ check failed or PSI > 0.2 |
| action_taken | STRING | One of: `none`, `alerted`, `retrain_triggered` |

The first write creates the table; subsequent writes append. Because we
own MODIFY on `student_work`, this is fully self-service.

In [ ]:
audit_schema = StructType([
    StructField("run_id", StringType()),
    StructField("run_timestamp", TimestampType()),
    StructField("rows_processed", LongType()),
    StructField("fraud_rate", DoubleType()),
    StructField("psi_score", DoubleType()),
    StructField("drift_detected", BooleanType()),
    StructField("action_taken", StringType()),
])

demo_row = [(
    str(uuid.uuid4()),
    datetime.now(timezone.utc),
    int(snap["rows_arrived"]),
    float(snap["fraud_rate_recent"]),
    0.05,        # placeholder PSI, we compute the real one in Lab 3
    False,
    "none",
)]

(
    spark.createDataFrame(demo_row, schema=audit_schema)
    .write.mode("append")
    .saveAsTable(AUDIT_TABLE)
)

display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(5))

## Lab 3 - Compute PSI on `amount` and write a real audit row (10 minutes)

PSI (Population Stability Index) on a continuous column is computed by:

1. Bucketing the baseline distribution into N (use 10) quantile bins.
2. Computing the proportion of rows in each bin for baseline and recent.
3. PSI = sum_i (recent_i - baseline_i) * ln(recent_i / baseline_i),
   with epsilon = 1e-6 to avoid divide-by-zero.

Implement:

- `psi_amount(df_baseline, df_recent, n_bins=10) -> float`
- `write_audit_row(snap, psi, drift_detected, action_taken)` that appends
  one row to `AUDIT_TABLE` using `audit_schema`.

Then call them. Use the full source table as the baseline and the CDF
inserts slice as the recent population.

In [ ]:
def psi_amount(df_baseline, df_recent, n_bins=10):
    # YOUR CODE
    return None

def write_audit_row(snap, psi, drift_detected, action_taken):
    # YOUR CODE
    pass

baseline_df = None  # YOUR CODE
recent_df = None    # YOUR CODE
psi_value = None    # YOUR CODE

drift = None        # YOUR CODE
write_audit_row(snap, psi_value, drift, "none")
display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(3))

In [ ]:
# SAFETY-NET for Lab 3.
if 'psi_value' not in dir() or psi_value is None:
    import math
    print("Using Lab 3 safety-net.")

    def psi_amount(df_baseline, df_recent, n_bins=10):
        qs = [i / n_bins for i in range(1, n_bins)]
        edges = df_baseline.approxQuantile("amount", qs, 0.001)
        edges = [float("-inf")] + edges + [float("inf")]

        def bucket_props(df):
            cases = F.when(F.col("amount") < edges[1], 0)
            for i in range(1, len(edges) - 1):
                cases = cases.when(F.col("amount") < edges[i + 1], i)
            df2 = df.withColumn("_b", cases)
            total = df2.count() or 1
            props = (
                df2.groupBy("_b").count()
                .withColumn("p", F.col("count") / F.lit(total))
                .select("_b", "p").collect()
            )
            return {r["_b"]: r["p"] for r in props}

        b = bucket_props(df_baseline)
        r = bucket_props(df_recent)
        eps = 1e-6
        psi = 0.0
        for i in range(n_bins):
            bp = b.get(i, 0.0) + eps
            rp = r.get(i, 0.0) + eps
            psi += (rp - bp) * math.log(rp / bp)
        return float(psi)

    def write_audit_row(snap, psi, drift_detected, action_taken):
        row = [(
            str(uuid.uuid4()),
            datetime.now(timezone.utc),
            int(snap["rows_arrived"]),
            float(snap["fraud_rate_recent"]),
            float(psi),
            bool(drift_detected),
            str(action_taken),
        )]
        (
            spark.createDataFrame(row, schema=audit_schema)
            .write.mode("append")
            .saveAsTable(AUDIT_TABLE)
        )

    baseline_df = spark.table(SOURCE_TABLE)
    recent_df = (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", start_v)
        .table(SOURCE_TABLE)
        .filter(F.col("_change_type") == "insert")
    )
    psi_value = psi_amount(baseline_df, recent_df)
    drift = (not all_passed) or (psi_value > 0.2)
    write_audit_row(snap, psi_value, drift, "none")
    print("psi_value =", round(psi_value, 4), "drift =", drift)

## Think About It

You now write to BOTH MLflow and a Delta audit table. That feels redundant.
When is each one the right home?

(Hint: MLflow is shaped around RUNS with parameters/metrics/artifacts and
shines for experiment comparison. A Delta audit table is shaped around
ROWS you can JOIN against transactions, customer data, or alert history.
Operationally you usually need both.)

## Topic 4 - Triggering retraining via the Databricks Workflows REST API

Detecting drift without acting on it is theater. The data engineer's job is
to hand off cleanly: when our signals fire, we start the ML engineer's
retraining job programmatically.

### The call

`POST {DATABRICKS_HOST}/api/2.1/jobs/run-now`

Headers: `Authorization: Bearer {DATABRICKS_TOKEN}`,
         `Content-Type: application/json`.

Body:
```json
{
  "job_id": 123456789,
  "notebook_params": {
    "trigger_reason": "drift_detected",
    "psi_score": "0.31",
    "audit_run_id": "..."
  }
}
```

Response (on success): `{ "run_id": ..., "number_in_job": ... }`. We
record the `run_id` in our audit table so the ML engineer can follow the
chain back from a retraining run to the data signals that caused it.

For class, the instructor has pre-created a tiny "stub" Databricks Job
that just prints its parameters; the job id is in the
`aws-course-creds` scope as `retrain-job-id`.

In [ ]:
RETRAIN_JOB_ID = int(dbutils.secrets.get(scope="aws-course-creds", key="retrain-job-id"))

def trigger_retraining(job_id, payload_params):
    url = f"{DATABRICKS_HOST}/api/2.1/jobs/run-now"
    body = {"job_id": job_id, "notebook_params": payload_params}
    r = requests.post(
        url,
        headers={
            "Authorization": f"Bearer {DATABRICKS_TOKEN}",
            "Content-Type": "application/json",
        },
        data=json.dumps(body),
        timeout=20,
    )
    r.raise_for_status()
    return r.json()

resp = trigger_retraining(
    RETRAIN_JOB_ID,
    {"trigger_reason": "demo", "psi_score": "0.0", "audit_run_id": "demo"},
)
print(resp)

## Lab 4 - Glue everything (15 minutes)

Implement `pipeline_run()` that ties the four topics together:

1. Compute the freshness snapshot (reuse `freshness_snapshot`).
2. Run the quality gate (reuse `run_quality_gate`).
3. Compute PSI on `amount` (reuse `psi_amount`).
4. Decide an `action_taken`:
   - `none` if all checks passed and PSI <= 0.2.
   - `alerted` if any check failed OR PSI > 0.2 but PSI <= 0.4.
   - `retrain_triggered` if PSI > 0.4 OR more than one gate check failed.
5. If `action_taken == "retrain_triggered"`, call `trigger_retraining`
   with `trigger_reason="drift_detected"` and the PSI score, then record
   the returned `run_id` in the audit row's `action_taken` string as
   `f"retrain_triggered:{run_id}"`.
6. Write the audit row.

Return the final dict.

You can call your own `pipeline_run()` at the end of the cell to see it
behave end-to-end. Because the source table is mostly stable, you will
typically see `action_taken="none"`. That is correct; the alerting and
retraining branches are exercised by the homework data.

In [ ]:
def pipeline_run() -> dict:
    # YOUR CODE
    return None

final = pipeline_run()
print(final)
display(spark.table(AUDIT_TABLE).orderBy(F.desc("run_timestamp")).limit(5))

## Recap

You shipped four pipeline-side MLOps building blocks today:

- **Freshness monitor** with Delta CDF, logged to MLflow.
- **Quality gate** with four PySpark assertions, halts on bad data.
- **Audit log** in Delta at `bread_academy.student_work.pipeline_audit_log`.
- **Auto retraining trigger** via Databricks Workflows REST API.

Notice what you did NOT do: you did not retrain the model, you did not
touch SageMaker, you did not edit the model code. As the data engineer
you OWNED the pipeline observability and HANDED OFF to the model owner.
That separation of duties is what makes MLOps survive contact with
production.

## Homework (async, ~45 min)

1. Add a fifth quality check: schema check. Read the schema of the source
   table once and store it as a tuple of (name, dtype) pairs. On every
   run, assert that the current schema is identical. Add it to
   `run_quality_gate`.
2. Compute PSI on a categorical column (`merchant_category`). PSI on
   categorical is the same formula but bins are category values, not
   quantiles. Add this as a second drift signal alongside `psi_amount`.
3. Build a small SQL query against `pipeline_audit_log` that returns the
   last 7 days of runs, with one row per day showing `n_runs`,
   `n_drift_detected`, `n_retrain_triggered`, `avg_psi`. This is the kind
   of pane an on-call data engineer wants pinned on a dashboard.

## Further reading

- Delta Change Data Feed: docs.databricks.com/aws/en/delta/delta-change-data-feed
- Jobs API run-now: docs.databricks.com/api/workspace/jobs/runnow
- MLflow experiments in jobs: docs.databricks.com/aws/en/mlflow/experiments
- Population Stability Index reference: see the Week 19 reading list.